In [0]:
webhook_url = dbutils.secrets.get(scope = "monitoring", key = "slack_webhook")

In [0]:
dbutils.widgets.text("job_name", "medallion_dlt_breweries",  "Nome del Job")
dbutils.widgets.text("job_status", "SUCCESS", "Stato del Job")
dbutils.widgets.text("run_url", "https://dbc-ffab9e1e-aa5d.cloud.databricks.com", "URL del Job")

In [0]:
import requests
import json
from datetime import datetime
import urllib.request

# Step 1 — Leggi i widget
job_name   = dbutils.widgets.get("job_name")
job_status = dbutils.widgets.get("job_status").upper()
run_url    = dbutils.widgets.get("run_url")

# Step 2 — Emoji e colore (FUORI dal try!)
if job_status == "SUCCESS":
    emoji = "✅"
    color = "#36a64f"
else:
    emoji = "❌"
    color = "#e01e5a"

# Step 3 — Conteggi layer
try:
    count_bronze = spark.table("pipeline_breweries.bronze_breweries").count()
    count_silver = spark.table("pipeline_breweries.silver_breweries").count()
    count_gold   = spark.table("pipeline_breweries.gold_breweries").filter("__END_AT IS NULL").count()

    if job_status == "SUCCESS":
        details = (
            f"🥉 Bronze: {count_bronze} records | "
            f"🥈 Silver: {count_silver} records | "
            f"🥇 Gold (current): {count_gold} records"
        )
    else:
        details = "❗ Pipeline fallita — controlla i log nel link qui sotto"

except Exception as e:
    details = f"⚠️ Conteggi non disponibili: {str(e)}"

# ── Timestamp

ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Step 4 — Payload
payload = {
    "attachments": [
        {
            "color": color,
            "blocks": [
                {
                    "type": "header",
                    "text": {
                        "type": "plain_text",
                        "text": f"{emoji} Job: {job_name}"
                    }
                },
                {
                    "type": "section",
                    "fields": [
                        {"type": "mrkdwn", "text": f"*Status:*\n{job_status}"},
                        {"type": "mrkdwn", "text": f"*Timestamp:*\n{ts}"}
                    ]
                },
                {
                    "type": "section",
                    "text": {
                        "type": "mrkdwn",
                        "text": f"*Record counts:*\n{details}"
                    }
                },
                {
                    "type": "actions",
                    "elements": [
                        {
                            "type": "button",
                            "text": {"type": "plain_text", "text": "🔗 Apri Run Databricks"},
                            "url": run_url
                        }
                    ]
                }
            ]
        }
    ]
}



In [0]:
# Step 5 — Invia

try:
    data = json.dumps(payload).encode("utf-8")

    req = urllib.request.Request(
        url = webhook_url,
        data = data,
        headers = {"Content-Type": "application/json"},
        method = "POST"
    )

    with urllib.request.urlopen(req) as f:
        result = f.read().decode("utf-8")
        print(f"Slack Response: {result}")
except Exception as e:
    print(f"Errore nell'invio a Slack: {str(e)}")

dbutils.notebook.exit("OK")